## Classifier

General idea : Predict the danger level of a circulation accident

In [55]:
import geopandas as gpd
import pandas as pd
import sklearn

In [56]:
df = gpd.read_file("data/accidents.shz")
df

,DATE,HEURE,CAUSE,TUES,BLESSES_GR,BLESSES_LE,DOMMAGES_M,METEO,ETAT_ROUTE,EMPLACEMEN,...,VITESSE_MA,LOCALITE,PRIORITE,ECLAIRAGE,LUMIERE,TRAFIC,GRAVITE_VA,TYPE_VALEU,TYPE_COMPL,geometry
0,20110102230000,18991230160000,manque d'égards en changeant de voie,0,0,0,3000,beau,sèche,giratoire,...,50.0,à l'intérieur d'une localité,aucune intersection,en service,aurore / crépuscule,normale,Dommages matériels,Accident lors de dépassement ou lors d'un chan...,Collision lors du changement de voie vers la g...,POINT (2570908 1162681)
1,20110104230000,18991230123500,inadaptée aux conditions de la route (mouillée...,0,0,0,1000,beau,verglacée,virage,...,80.0,à l'extérieur d'une localité,aucune intersection,aucun,jour,faible,Dommages matériels,Collision frontale,Frôlement avec le trafic venant en sens invers...,POINT (2568472 1166525)
2,20110104230000,18991230211500,"arrêt (freinage) sans raison sur la chaussée, ...",0,0,0,5000,couvert,sèche,ligne droite,...,50.0,à l'extérieur d'une localité,aucune intersection,en service,nuit,faible,Dommages matériels,Accident par tamponnement,Collision avec un véhicule en marche,POINT (2579634 1183736)
3,20110105230000,18991230063000,inadaptée aux conditions de la route (mouillée...,0,0,0,1500,pluie givrante,verglacée,virage,...,50.0,à l'intérieur d'une localité,aucune intersection,en service,aurore / crépuscule,faible,Dommages matériels,Dérapage ou perte de maîtrise,Collision avec un obstacle hors de la chaussée,POINT (2575732 1201401)
4,20110106230000,18991230161000,non-respect du signal de priorité «cédez le pa...,0,0,0,5000,couvert,humide,intersection,...,80.0,à l'extérieur d'une localité,signal «cédez le passage»,aucun,nuit,forte,Dommages matériels,Accident en s'engageant sur une route,Collision en obliquant à gauche avec le trafic...,POINT (2580788 1201641)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18586,20171024220000,18991230110000,autre attitude fautive lors des mouvements dan...,0,0,0,2000,beau,sèche,place de stationnement,...,50.0,à l'intérieur d'une localité,aucune intersection,hors service,jour,faible,Dommages matériels,Accident lors du stationnement,Avec violation de l'obligation de déclarer,POINT (2578230 1183912)
18587,20180107230000,18991230030000,présence d'animaux sauvages sur la chaussée,0,0,0,2000,couvert,sèche,ligne droite,...,120.0,à l'extérieur d'une localité,aucune intersection,aucun,nuit,faible,Dommages matériels,Accident impliquant un animal,Collision avec du gibier (G),POINT (2559620 1153696)
18588,20230730220000,18991230114400,marche arrière imprudente,0,0,0,5000,beau,sèche,place de stationnement,...,50.0,à l'intérieur d'une localité,aucune intersection,hors service,jour,faible,Dommages matériels,Accident lors du stationnement,Collision avec un obstacle fixe (y compris véh...,POINT (2569312 1156707)
18589,20231107230000,18991230100000,cause inconnue,0,0,0,500,beau,sèche,giratoire,...,50.0,à l'intérieur d'une localité,aucune intersection,hors service,jour,normale,Dommages matériels,Dérapage ou perte de maîtrise,Collision avec un autre usager (piéton y compris),POINT (2578217 1185451)


In [57]:
print(df.columns)
df.shape

Index(['DATE', 'HEURE', 'CAUSE', 'TUES', 'BLESSES_GR', 'BLESSES_LE',
       'DOMMAGES_M', 'METEO', 'ETAT_ROUTE', 'EMPLACEMEN', 'GENRE_ROUT',
       'VITESSE_MA', 'LOCALITE', 'PRIORITE', 'ECLAIRAGE', 'LUMIERE', 'TRAFIC',
       'GRAVITE_VA', 'TYPE_VALEU', 'TYPE_COMPL', 'geometry'],
      dtype='str')


(18591, 21)

In [58]:
df.dtypes

DATE               str
HEURE              str
CAUSE              str
TUES             int64
BLESSES_GR       int64
BLESSES_LE       int64
DOMMAGES_M       int64
METEO              str
ETAT_ROUTE         str
EMPLACEMEN         str
GENRE_ROUT         str
VITESSE_MA     float64
LOCALITE           str
PRIORITE           str
ECLAIRAGE          str
LUMIERE            str
TRAFIC             str
GRAVITE_VA         str
TYPE_VALEU         str
TYPE_COMPL         str
geometry      geometry
dtype: object

In [59]:
df.isna().sum()
df.dropna(inplace=True)
df.isna().sum()

DATE          0
HEURE         0
CAUSE         0
TUES          0
BLESSES_GR    0
BLESSES_LE    0
DOMMAGES_M    0
METEO         0
ETAT_ROUTE    0
EMPLACEMEN    0
GENRE_ROUT    0
VITESSE_MA    0
LOCALITE      0
PRIORITE      0
ECLAIRAGE     0
LUMIERE       0
TRAFIC        0
GRAVITE_VA    0
TYPE_VALEU    0
TYPE_COMPL    0
geometry      0
dtype: int64

In [60]:

#Drop columns with informations we could not get
#TODO : SEE AGAINM WHICH COLUMNS TO DROP; MIGHT BE KEEP SOME OF THEM BY USING THE INFOS OF RETRANSCRIPTION ?
COLUMNS_TO_DROP = ['ECLAIRAGE', 'ETAT_ROUTE', 'DOMMAGES_M', 'PRIORITE', 'TYPE_VALEU', 'TYPE_COMPL', 'GRAVITE_VA']

if all(col in df.columns for col in COLUMNS_TO_DROP):
    df.drop(columns=COLUMNS_TO_DROP, inplace=True)

df.describe()


,TUES,BLESSES_GR,BLESSES_LE,VITESSE_MA
count,14311.000000,14311.000000,14311.000000,14311.000000
mean,0.009573,0.146111,0.589896,64.471595
std,0.107603,0.389187,0.796046,23.553644
min,0.000000,0.000000,0.000000,5.000000
25%,0.000000,0.000000,0.000000,50.000000
50%,0.000000,0.000000,0.000000,50.000000
75%,0.000000,0.000000,1.000000,80.000000
max,4.000000,5.000000,15.000000,120.000000


In [61]:
#get unique values for each column
#REMOVED FROM THE LIST : 'ETAT_ROUTE', 'PRIORITE', 'ECLAIRAGE', , 'TYPE_VALEU', 'TYPE_COMPL', 'GRAVITE_VA'
UNIQUE_COLUMNS_TO_CHECK = ['METEO', 'EMPLACEMEN', 'GENRE_ROUT', 'LOCALITE',  'LUMIERE', 'TRAFIC']
for col in UNIQUE_COLUMNS_TO_CHECK:
    print(f"Unique values for column {col}: {df[col].unique()}\n")

Unique values for column METEO: <StringArray>
[          'beau',        'couvert', 'pluie givrante',          'autre',
          'pluie', 'chute de neige',          'grêle']
Length: 7, dtype: str

Unique values for column EMPLACEMEN: <StringArray>
[             'giratoire',                 'virage',           'ligne droite',
           'intersection', 'place de stationnement',                  'place',
                  'autre',          'aire de repos']
Length: 8, dtype: str

Unique values for column GENRE_ROUT: <StringArray>
[                 'route principale',                  'route secondaire',
                         'autoroute',                             'autre',
 'installation annexe sur autoroute',                    'semi-autoroute']
Length: 6, dtype: str

Unique values for column LOCALITE: <StringArray>
['à l'intérieur d'une localité', 'à l'extérieur d'une localité']
Length: 2, dtype: str

Unique values for column LUMIERE: <StringArray>
['aurore / crépuscule', 'jour', 'n

In [67]:
from sklearn.preprocessing import OneHotEncoder
#convert date to datetime
df_cleaned = df.copy()
df_cleaned['DATE'] = pd.to_datetime(df['DATE'].str[:8], format='%Y%m%d')
heure_str = df['HEURE'].str[-6:]
df_cleaned['HEURE'] = pd.to_datetime(heure_str, format='%H%M%S').dt.time


df_cleaned.drop(columns=['DATE', 'HEURE', 'CAUSE'], inplace=True)

categorical_cols = df_cleaned.select_dtypes(include=['object']).columns

encoder = OneHotEncoder(sparse_output=False, drop='first')
encoded_features = encoder.fit_transform(df_cleaned[categorical_cols])

encoded_df = pd.DataFrame(
    encoded_features, 
    columns=encoder.get_feature_names_out(categorical_cols),
    index=df_cleaned.index
)

df_final = pd.concat([df_cleaned.drop(columns=categorical_cols), encoded_df], axis=1)
df_final


C:\Users\gaeta\AppData\Local\Temp\ipykernel_13916\1832201914.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df_cleaned.select_dtypes(include=['object']).columns


,TUES,BLESSES_GR,BLESSES_LE,VITESSE_MA,geometry,METEO_beau,METEO_chute de neige,METEO_couvert,METEO_grêle,METEO_pluie,...,GENRE_ROUT_semi-autoroute,LOCALITE_à l'intérieur d'une localité,LUMIERE_inconnu,LUMIERE_jour,LUMIERE_nuit,TRAFIC_faible,TRAFIC_file arrêtée,TRAFIC_file ralentie,TRAFIC_forte,TRAFIC_normale
0,0,0,0,50.0,POINT (2570908 1162681),1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,0,0,0,80.0,POINT (2568472 1166525),1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
2,0,0,0,50.0,POINT (2579634 1183736),0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
3,0,0,0,50.0,POINT (2575732 1201401),0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,0,0,0,80.0,POINT (2580788 1201641),0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18586,0,0,0,50.0,POINT (2578230 1183912),1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
18587,0,0,0,120.0,POINT (2559620 1153696),0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
18588,0,0,0,50.0,POINT (2569312 1156707),1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
18589,0,0,0,50.0,POINT (2578217 1185451),1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


In [70]:
#TODO: SEE IF WE COULD KEEP IT FOR FURTHER VISUALIZATION....
df_final.drop('geometry', inplace=True, axis=1)

In [ ]:
#train_val_test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df_final.drop(columns=['TUES','BLESSES_GR','BLESSES_LE']), df_final[['TUES', 'BLESSES_GR', 'BLESSES_LE']], test_size=0.4, random_state=42)


In [64]:
from sklearn.model_selection import GridSearchCV

def find_best_hyperparameters(regressor, param_grid, X_train, y_train, scoring = 'r2'):
    grid_search = GridSearchCV(regressor, param_grid, cv=5, scoring=scoring, return_train_score=True, n_jobs=-1, verbose=2) # use all the cores of the CPU
    grid_search.fit(X_train, y_train)
    print("Best hyperparameters: {}".format(grid_search.best_params_))
    print("Best score (r2): {:.2f}".format(grid_search.best_score_))
    return grid_search.best_estimator_

In [65]:
from sklearn.model_selection import cross_validate
def print_cross_validation_results(regressor, X_train, y_train):
    # cross validate the model
    scoring = ['r2', 'neg_mean_absolute_error', 'neg_root_mean_squared_error']
    cv_results = cross_validate(regressor, X_train, y_train, cv=5, scoring = scoring, n_jobs=-1, return_train_score=True)
    # Print the results
    print("Train R^2: {:.2f}".format(cv_results['train_r2'].mean()))
    print("Validation R^2: {:.2f}".format(cv_results['test_r2'].mean()))
    print("Train MAE: {:.2f}".format(-cv_results['train_neg_mean_absolute_error'].mean()))
    print("Validation MAE: {:.2f}".format(-cv_results['test_neg_mean_absolute_error'].mean()))
    print("Train RMSE: {:.2f}".format(-cv_results['train_neg_root_mean_squared_error'].mean()))
    print("Validation RMSE: {:.2f}".format(-cv_results['test_neg_root_mean_squared_error'].mean()))

In [ ]:
from sklearn.ensemble import RandomForestRegressor

param_grid = {'n_estimators': [10, 100], 'max_depth': [None, 1, 3], 'min_samples_split': [2, 4], 'min_samples_leaf': [1]}

rf_reg = RandomForestRegressor()

rf_reg = find_best_hyperparameters(rf_reg, param_grid, X_train, y_train)

print_cross_validation_results(rf_reg, X_train, y_train)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best hyperparameters: {'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
Best score (r2): 0.01
Train R^2: 0.02
Validation R^2: 0.01
Train MAE: 0.29
Validation MAE: 0.29
Train RMSE: 0.43
Validation RMSE: 0.43
